# SEC XBRL Financial Statement Analysis

**Prepared for:** Research Manager  
**Company:** Apple Inc. (CIK 320193)  
**Dataset:** SEC Financial Statement Data Sets, 2026 Q2 (latest posted quarter as of October 2, 2026)

This analysis reconstructs flattened primary statement facts for Apple's latest 10-K or 10-Q in the selected SEC dataset. It presents reported amounts and common-size percentages, then compares nonzero financial-statement amounts with Benford's expected first-digit distribution. The SEC dataset is an analytical aid; the filing linked below is the source to consult for the exact rendered statement and reporting context.

## 1. Retrieve the SEC dataset
The SEC updates these files quarterly. This notebook downloads the latest posted quarter directly from the [SEC Financial Statement Data Sets page](https://www.sec.gov/dera/data/financial-statement-data-sets.html). The ZIP is approximately 58 MB; it is saved locally and is not included in the helper-code archive. SEC requests should identify the user with a valid contact email.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import urllib.request

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'xbrl_fs').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in __import__('sys').path:
    __import__('sys').path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)
DATASET_FILE = DATA_DIR / '2026q2.zip'
DATASET_URL = 'https://www.sec.gov/files/dera/data/financial-statement-data-sets/2026q2.zip'
USER_AGENT = 'Student financial research your.name your.email@example.com'  # Replace with your name and contact email.

if not DATASET_FILE.exists():
    request = urllib.request.Request(DATASET_URL, headers={'User-Agent': USER_AGENT})
    with urllib.request.urlopen(request) as response, DATASET_FILE.open('wb') as output:
        output.write(response.read())

print(f'Dataset: {DATASET_FILE.name} ({DATASET_FILE.stat().st_size / 1_000_000:.1f} MB)')

## 2. Load the SEC tables and select the filing
`SUB` identifies each filing, `NUM` contains reported numeric facts, and `TAG` supplies human-readable concept labels. The statement helper selects the most recently filed 10-K or 10-Q for the chosen CIK and excludes segment-specific rows when that field is available.

In [ ]:
from xbrl_fs import create_financial_statements

with ZipFile(DATASET_FILE) as archive:
    sec_tables = {}
    for table_name in ("sub", "num", "tag"):
        member = next(name for name in archive.namelist() if name.rsplit("/", 1)[-1].lower() == f"{table_name}.tsv")
        with archive.open(member) as file:
            sec_tables[table_name] = pd.read_csv(file, sep="\t", dtype=str, low_memory=False)

submissions = sec_tables["sub"]
numeric = sec_tables["num"]
tags = sec_tables["tag"]

CIK = "320193"  # Apple Inc.
filing, statements = create_financial_statements(submissions, numeric, tags, CIK)
company = filing.get("name", "Company")
print(f"{company} | CIK {int(CIK):010d} | Form {filing['form']} | Filed {filing['filed']} | Period ended {filing.get('period', 'not reported')}")
accession = str(filing["adsh"]).replace("-", "")
filing_url = f"https://www.sec.gov/Archives/edgar/data/{int(CIK)}/{accession}/{filing['adsh']}-index.htm"
print(f"SEC filing index: {filing_url}")
print({name: len(table) for name, table in statements.items()})

## 3. Reported financial statements
The SEC extract is flattened, so rows are grouped into balance sheet (instant facts), income statement, and cash flow statement (duration facts identified by cash-flow concepts). `quarters` preserves SEC's duration indicator, which distinguishes quarterly from year-to-date amounts. The presentation below shows the relevant facts as extracted; confirm line order, signs, units, and comparative columns against the linked filing.

In [ ]:
statement_titles = {
    'balance_sheet': 'Balance Sheet',
    'income_statement': 'Income Statement',
    'cash_flow': 'Cash Flow Statement',
}
for name, title in statement_titles.items():
    print(f'\n{title} (reported values)')
    display(statements[name].head(35).style.format({'value': '{:,.0f}'}))

## 4. Common-size statements
Balance sheet amounts are divided by total assets. Income statement and cash-flow amounts are divided by revenue for the same period and duration when a matching revenue fact is available. Values are percentages; rows without a defensible denominator are omitted.

In [ ]:
from xbrl_fs import common_size_statements

common_size = common_size_statements(statements)
for name, title in statement_titles.items():
    print(f'\n{title} (common-size percentages)')
    display(common_size[name].head(35).style.format({'value': '{:,.2f}%'}))

## 5. Benford's Law assessment
The analysis uses the first nonzero digit of absolute reported amounts. Zeroes and values without a usable digit are ignored. Since the helper's MAD is expressed in percentage points, it is converted to a proportion before applying the commonly cited MAD bands. This is a screening comparison, not a fraud test: statement values are selected, related, and often rounded, so departures require accounting and filing context.

In [ ]:
from benford_analysis import analyze_benford

benford_results = {}
for name, title in statement_titles.items():
    values = statements[name]['value'].abs()
    comparison, mad, ks = analyze_benford(values)
    benford_results[name] = {'comparison': comparison, 'mad_pp': mad, 'ks_pp': ks, 'n': int(comparison['actual_percent'].sum() > 0 and values.ne(0).sum())}
    print(f'{title}: {benford_results[name]["n"]:,} nonzero values | MAD {mad:.4f} percentage points | KS {ks:.4f} percentage points')
    display(comparison.style.format({'actual_percent': '{:.2f}%', 'benford_percent': '{:.2f}%'}))

## 6. Results and interpretation
The following text is generated from the observed sample. MAD bands are screening conventions, not statistical proof; KS is reported descriptively because sample size and dependence between accounting facts limit a simple pass/fail interpretation.

In [ ]:
for name, title in statement_titles.items():
    result = benford_results[name]
    largest = result["comparison"].assign(
        deviation=lambda data: (data["actual_percent"] - data["benford_percent"]).abs()
    ).nlargest(3, "deviation")
    digits = ", ".join(str(int(digit)) for digit in largest["digit"])
    mad_ratio = result["mad_pp"] / 100
    if mad_ratio < 0.006:
        interpretation = "close conformity under the commonly cited MAD screening bands"
    elif mad_ratio < 0.012:
        interpretation = "acceptable conformity under the commonly cited MAD screening bands"
    elif mad_ratio < 0.015:
        interpretation = "marginal conformity under the commonly cited MAD screening bands"
    else:
        interpretation = "non-conformity under the commonly cited MAD screening bands; investigate context before drawing conclusions"
    print(
        f"{title}: {interpretation}. The largest digit deviations are {digits}; "
        f"MAD is {result['mad_pp']:.4f} percentage points and "
        f"KS is {result['ks_pp']:.4f} percentage points."
    )

print(
    "Conclusion: These results identify digit patterns for follow-up only. "
    "They do not establish error or misconduct. Review unusual items in the "
    "original SEC filing and assess materiality, accounting policy, and the "
    "small, non-independent sample before reaching a research conclusion."
)

## Method and source notes
- SEC data source: [Financial Statement Data Sets](https://www.sec.gov/dera/data/financial-statement-data-sets.html); 2026 Q2 ZIP, accessed October 2, 2026.
- Dataset documentation and the SEC disclaimer: [Financial Statement Data Sets documentation](https://www.sec.gov/files/financial-statement-data-sets.pdf). The SEC cautions that extracts may contain errors and are not a substitute for full filings.
- Helper functions are in `xbrl_fs/statements.py` and `benford_analysis.py`; their docstrings include doctests. Run `python -m doctest xbrl_fs/statements.py benford_analysis.py` from the project directory to check them.
- The classifier reconstructs statements from the flattened data and may not perfectly preserve SEC display order or all filing-specific presentation choices. Verify the final tables against the linked filing before using them in a research report.